# 04 — OCR Dataset Creation

Extract OCR crops from detector labels and produce `labels.csv` (image,text).

In [ ]:
import csv
from pathlib import Path

import cv2
from tqdm import tqdm

In [ ]:
DATASETS = [
    Path('C:/path/to/private/data/Bootstrap_5k'),
    Path('C:/path/to/private/data/Bootstrap_5k_v2'),
]
OCR_ROOT = Path('C:/path/to/private/data/OCR_Dataset')
OCR_IMAGES = OCR_ROOT / 'images'
CSV_FILE = OCR_ROOT / 'labels.csv'
OCR_IMAGES.mkdir(parents=True, exist_ok=True)

rows = []
crop_id = 0
for dataset_root in DATASETS:
    for image_rel, label_rel in [('train/images', 'train/labels'), ('val/images', 'val/labels')]:
        image_dir = dataset_root / image_rel
        label_dir = dataset_root / label_rel
        if not image_dir.exists():
            continue
        image_files = [p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
        for image_path in tqdm(image_files, desc=f'{dataset_root.name}/{image_rel}'):
            label_path = label_dir / f'{image_path.stem}.txt'
            if not label_path.exists():
                continue
            lines = [x.strip() for x in label_path.read_text().splitlines() if x.strip()]
            if not lines:
                continue
            img = cv2.imread(str(image_path))
            if img is None:
                continue
            h, w = img.shape[:2]
            parts = lines[0].split()
            if len(parts) < 5:
                continue
            _, xc, yc, bw, bh = map(float, parts[:5])
            x1 = max(0, int((xc - bw / 2) * w))
            y1 = max(0, int((yc - bh / 2) * h))
            x2 = min(w, int((xc + bw / 2) * w))
            y2 = min(h, int((yc + bh / 2) * h))
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            plate_text = image_path.stem.rsplit('_', 1)[0].upper()
            crop_name = f'{crop_id:06d}.jpg'
            cv2.imwrite(str(OCR_IMAGES / crop_name), crop)
            rows.append([crop_name, plate_text])
            crop_id += 1

with open(CSV_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['image', 'text'])
    writer.writerows(rows)

print('total_crops =', len(rows))
print('saved =', CSV_FILE)